# GVH Diagonal Cubic — CHH-05
## Couplage et équations du mouvement : audit variationnel avec verrou matière

**Classification : complément scientifique CHH, non canonique.** Cette étape consulte le véritable candidat $S_{\rm cand}$ tel qu'archivé dans `.3.3.28` et le checkpoint `.3.3.28.1`, sans les réécrire ni substituer une action phénoménologique.

**Question contrôlée :** peut-on dériver les couplages de deux sous-cubes hétérogènes et leur activité interne des seuls termes présents dans $S_{\rm cand}$ ?

**Réponse attendue à tester :** l'action peut donner une variation du secteur de cisaillement; elle ne détermine pas automatiquement les flux de matière de sous-cubes distincts car les degrés de liberté de matière et leurs couplages ne sont pas spécifiés.

Distinguer strictement : `EXACT_ACTION_VARIATION`, `ANZATZ_CONDITIONAL`, `STANDARD_PHYSICS_ADDED_SECTOR`, `NOT_DERIVED` et `BLOCKED`.


# Niveau 1 — GVH : action candidate archivée

La source canonique `.3.3.28` utilise la signature $(-+++)$, $u^\mu u_\mu=-1$ et un trièdre auxiliaire $e_A$. Son action de travail est

$$
S_{\rm cand}=\frac{1}{16\pi G_0}\int d^4x\sqrt{-g}\left[R+\mathcal L_u+\lambda_u(u^2+1)+\frac{\zeta_C}{2}\mathcal O_C+\sum_A\eta_A(u\cdot e_A)+\frac12\sum_{AB}\eta_{AB}(e_A\cdot e_B-\delta_{AB})\right].
$$

Ici $\mathcal O_C=\sum_{A=1}^3s_A^2$, $s_A=e_A^\mu e_A^\nu\sigma_{\mu\nu}$ et $\sigma$ est le cisaillement spatial sans trace de $u$. L'action possède des EOM variationnelles en $(g,u,e)$ mais **pas de secteur matière $S_m$, ni de variables spécifiques $C_i$, ni de courant de masse de chaque sous-cube, ni de variable d'activité $A_C$**.

Le trièdre est auxiliaire, sans terme cinétique $\nabla e_A$. L'EOM du vecteur archivée comporte un courant cubique dérivé; son reste algébrique et l'EOM métrique ne sont pas tous deux complètement développés en composantes sur un fond physique général.

**Source et provenance :** les empreintes figées ci-dessous seront comparées, lorsque les fichiers originaux sont disponibles; une copie autonome peut néanmoins tourner sur Colab sans ces archives montées.


In [1]:
import numpy as np, sympy as sp, math, json, hashlib
from pathlib import Path
from scipy.integrate import solve_ivp
from scipy.linalg import eigh
from scipy.optimize import minimize_scalar
from IPython.display import display

STEM='GVH_CHH_05_Action_Coupling_and_EOM_Gate_FAST'
KNOWN_SOURCE_SHA = {'GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb': {'sha256': 'c8015f1d771caa5f65c8ad5d64505733ebfa96a81f6153a8b1d8c2233cfbeefa', 'bytes': 49860}, 'GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.1_Cubic_Vector_Closure_Metric_Helix_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb': {'sha256': '0018b54c8f07984e3ebf35ea6989e6772bebb815492b66b672bed83241f8d5b4', 'bytes': 48200}, 'GVH_Action_Candidate_Two_Cube_Shear_Overlap_Gate_FAST_results.json': {'sha256': 'b56081da5af9343b44032fd17ef75c89df18bacd6a776f0f5ec21b052e3716a9', 'bytes': 3080}, 'GVH_CHH_04_Heterogeneous_Aggregation_Benchmark_FAST_results.json': {'sha256': 'bed5b76f41b2d7d089f819f059e66ce038e7f9cb26f48f897350a5c5cf00e780', 'bytes': 6874}}
search_dirs=[Path('/mnt/data'),Path.cwd(),Path('/content')]
upstream_audit={}
for fname, expected in KNOWN_SOURCE_SHA.items():
    found=next((d/fname for d in search_dirs if (d/fname).is_file()),None)
    if found is None:
        found=next((d/'gvh_source'/fname for d in search_dirs if (d/'gvh_source'/fname).is_file()),None)
    if found:
        digest=hashlib.sha256(found.read_bytes()).hexdigest()
        upstream_audit[fname] = 'PRESENT_MATCH' if digest==expected['sha256'] else 'PRESENT_MISMATCH'
        assert digest==expected['sha256'], f'Provenance mismatch: {fname}'
    else: upstream_audit[fname]='NOT_MOUNTED_PORTABLE_REFERENCE'
print('Provenance audit:',json.dumps(upstream_audit,indent=2))
CHECKS={}
def check(name,condition):
    CHECKS[name]=bool(condition)
    assert CHECKS[name], f'FAILED: {name}'


Provenance audit: {
  "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb": "PRESENT_MATCH",
  "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.1_Cubic_Vector_Closure_Metric_Helix_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb": "PRESENT_MATCH",
  "GVH_Action_Candidate_Two_Cube_Shear_Overlap_Gate_FAST_results.json": "PRESENT_MATCH",
  "GVH_CHH_04_Heterogeneous_Aggregation_Benchmark_FAST_results.json": "PRESENT_MATCH"
}


## 1.1 Terme croisé algébrique de cisaillement

**Condition explicitement ajoutée**, pas déduite des EOM : choisir localement un même trièdre $(e_A)$ et une décomposition $s_A=s_A^{(1)}+s_A^{(2)}$. Il vient

$$
\Delta\mathcal L_{C,12}=\frac{\zeta_C}{2}\left[\sum_A(s_A^{(1)}+s_A^{(2)})^2-\sum_A(s_A^{(1)})^2-\sum_A(s_A^{(2)})^2\right]
=\zeta_C\sum_A s_A^{(1)}s_A^{(2)}.
$$

Ce terme est une **identité algébrique sur l'ansatz**. Il ne suffit ni à identifier une force entre corps, ni à établir l'indépendance physique des deux parties. La décomposition du cisaillement est une donnée supplémentaire. Nous vérifierons si le terme « entre cubes » dépend arbitrairement de cette décomposition.


In [2]:
p,q,d,z=sp.symbols('p q delta zeta_C',real=True)
s1=sp.Matrix([p,-p,0]); s2=sp.Matrix([q,-q,0]); s_total=s1+s2
Lc=lambda s:sp.expand(z*sum(t*t for t in s)/2)
cross=sp.simplify(Lc(s_total)-Lc(s1)-Lc(s2))
check('cubic_cross_expansion_exact',sp.simplify(cross-2*z*p*q)==0)
# Re-label two shear portions while preserving the *same total shear*.
relabel1=sp.Matrix([p+d,-p-d,0]); relabel2=sp.Matrix([q-d,-q+d,0])
newcross=sp.simplify(Lc(relabel1+relabel2)-Lc(relabel1)-Lc(relabel2))
check('fixed_total_shear_under_repartition',sp.simplify(relabel1+relabel2-s_total)==sp.zeros(3,1))
check('total_cubic_density_partition_invariant',sp.simplify(Lc(relabel1+relabel2)-Lc(s_total))==0)
check('cross_density_partition_dependent',sp.simplify(newcross-cross)!=0)
relabel_witness={
 'total_density':float(Lc(s_total).subs({p:.6,q:.4,z:.7})),
 'cross_before':float(cross.subs({p:.6,q:.4,z:.7})),
 'cross_after':float(newcross.subs({p:.6,q:.4,d:.2,z:.7})),
}
print('Delta L cross=',cross,'repartition Delta=',sp.factor(newcross-cross))
print('Witness:',relabel_witness)


Delta L cross= 2*p*q*zeta_C repartition Delta= -2*delta*zeta_C*(delta + p - q)
Witness: {'total_density': 0.7, 'cross_before': 0.33599999999999997, 'cross_after': 0.22400000000000006}


## 1.2 Variation réellement admissible du trièdre

Le checkpoint `.3.3.28` déduit la contrainte d'orientation

$$\boxed{\zeta_C(s_A-s_B)M_{AB}=0}\qquad(A<B).$$

Nous la retrouvons directement, indépendamment, en faisant tourner infinitésimalement les axes $1,2$ à $u$ et $g$ fixés et en calculant $\frac{d}{d\theta}\mathcal L_C$.

Attention : une variation **du trièdre** n'est pas une variation libre des $s_A$ et n'est pas une EOM pour la masse d'un corps céleste.


In [3]:
a,b,h,th=sp.symbols('a b h theta',real=True)
M=sp.Matrix([[a,h,0],[h,b,0],[0,0,-a-b]]) # STF
R=sp.Matrix([[sp.cos(th),-sp.sin(th),0],[sp.sin(th),sp.cos(th),0],[0,0,1]])
Mr=sp.simplify(R.T*M*R)
Lrot=sp.expand(z*sum(Mr[i,i]**2 for i in range(3))/2)
dLrot=sp.simplify(sp.diff(Lrot,th).subs(th,0))
check('triad_orientation_variation_matches_archived',sp.simplify(dLrot-2*z*(a-b)*h)==0)
check('triad_eigenframe_is_stationary',sp.simplify(dLrot.subs(h,0))==0)
check('triad_zeta_zero_orientation_degenerate',sp.simplify(dLrot.subs(z,0))==0)
check('triad_equal_diagonal_stationary',sp.simplify(dLrot.subs(b,a))==0)
vals={a:.8,b:-.3,h:.12,z:.4}
for eps in [1e-3,1e-4,1e-5]:
    fun=sp.lambdify(th,Lrot.subs(vals),'numpy')
    fd=(fun(eps)-fun(-eps))/(2*eps)
    exact=float(dLrot.subs(vals))
    check('triad_variation_finite_difference_'+str(eps),abs(fd-exact)<(1e-4 if eps==1e-3 else 2e-7))
print('dL/dtheta at 0=',dLrot)


dL/dtheta at 0= 2*h*zeta_C*(a - b)


## 1.3 Pourquoi on ne peut pas additionner librement les $u_i^\mu$

La contrainte cinématique essentielle de l'action est $u^\mu u_\mu=-1$. Deux vecteurs temporels unitaires (pour deux sous-cubes) ne donnent **pas**, par simple addition, un nouveau vecteur unitaire. Même normalisée, leur somme ne définit pas sans autre hypothèse le champ de cisaillement global ni des EOM couplées. 

Ce contrôle démontre une obstruction explicite à la superposition naïve des champs de sous-cubes.


In [4]:
eta=sp.diag(-1,1,1,1)
vA,vB=sp.symbols('v_A v_B',real=True)
ua=sp.Matrix([1/sp.sqrt(1-vA*vA),vA/sp.sqrt(1-vA*vA),0,0])
ub=sp.Matrix([1/sp.sqrt(1-vB*vB),vB/sp.sqrt(1-vB*vB),0,0])
norm=lambda v:sp.simplify((v.T*eta*v)[0])
check('unit_timelike_A',sp.simplify(norm(ua)+1)==0)
check('unit_timelike_B',sp.simplify(norm(ub)+1)==0)
witness_sum=float(norm((ua+ub).subs({vA:.2,vB:-.35})))
check('naive_sum_not_unit_timelike',abs(witness_sum+1)>0.1)
check('same_u_sum_still_not_unit',sp.simplify(norm((ua+ua).subs(vA,0))+1)!=0)
print('uA+uB norm sample=',witness_sum)


uA+uB norm sample= -4.331602939976188


# Niveau 2 — Physique établie : une extension matière **distincte** et explicitement hypothétique

Pour comprendre **ce qu'il faudrait fournir** avant de prétendre prédire des interactions entre objets, examinons un simple secteur de matière classique, à deux espèces scalaires $\phi_A,\phi_B$. Ceci **n'est pas** dans $S_{\rm cand}$ et n'est pas « déduit de GVH » :

$$
S_m^{\rm example}=-\frac12\int d^4x\sqrt{-g}\left[\sum_{i=A,B}\left(g^{\mu\nu}\partial_\mu\phi_i\partial_\nu\phi_i+m_i^2\phi_i^2\right)+2\kappa\phi_A\phi_B\right].
$$

Les EOM dans un fond fixe sont

$$\Box\phi_A-m_A^2\phi_A-\kappa\phi_B=0,\qquad \Box\phi_B-m_B^2\phi_B-\kappa\phi_A=0.$$

Avec la convention $(-+++)$, en régime homogène cela devient $\ddot\phi_A+m_A^2\phi_A+\kappa\phi_B=0$, etc. Le coefficient $\kappa$ est **prescrit** : ces EOM démontrent comment se calcule un couplage *si on le postule*, pas qu'il existe dans l'action GVH archivée. Les masses $m_i$ et $\kappa$ de cette simulation sont adimensionnés; aucune fréquence SI de corps astrophysique n'en découle.


In [5]:
PhiA,PhiB,dPhiA,dPhiB,m2a,m2b,kappa=sp.symbols('Phi_A Phi_B dPhi_A dPhi_B m_A_sq m_B_sq kappa',real=True)
L=sp.Rational(1,2)*(dPhiA**2+dPhiB**2-m2a*PhiA**2-m2b*PhiB**2-2*kappa*PhiA*PhiB)
EL_A=sp.simplify(sp.diff(L,PhiA)) # - d/dt dL/d(dPhiA) on EOM => ddot A - ∂L/∂A
EL_B=sp.simplify(sp.diff(L,PhiB))
check('scalar_EL_A_mass_coupling',sp.simplify(EL_A+m2a*PhiA+kappa*PhiB)==0)
check('scalar_EL_B_mass_coupling',sp.simplify(EL_B+m2b*PhiB+kappa*PhiA)==0)
Umat=sp.Matrix([[m2a,kappa],[kappa,m2b]])
lambda_sym=sp.symbols('lambda',real=True)
char=sp.expand((Umat-lambda_sym*sp.eye(2)).det())
expected=(m2a-lambda_sym)*(m2b-lambda_sym)-kappa**2
check('normal_mode_characteristic',sp.simplify(char-expected)==0)
# Stable parameter region: kappa^2 < m_A^2 m_B^2 and positive squared masses
check('zero_coupling_independent_species',sp.simplify(Umat.subs(kappa,0)[0,1])==0)
print('EL couplings:',EL_A,EL_B,'characteristic:',char)


EL couplings: -Phi_A*m_A_sq - Phi_B*kappa -Phi_A*kappa - Phi_B*m_B_sq characteristic: -kappa**2 + lambda**2 - lambda*m_A_sq - lambda*m_B_sq + m_A_sq*m_B_sq


# Niveau 3 — Test numérique minimal indépendant

Comparer les EOM du secteur matière *ajouté* avec leur solution modale analytique, sur plusieurs conditions initiales réservées. Pour éviter toute validation circulaire, nous ne choisissons aucun paramètre pour correspondre aux données de CHH-01/04 ni aux observations ALMA. Nous vérifions la conservation d'énergie de cet exemple conservatif.


In [6]:
MA2,MB2,K=1.0,2.25,0.32  # adimensionné ; strictement prescrit, externe à S_cand
P=np.array([[MA2,K],[K,MB2]],dtype=float)
eigs,V=eigh(P)
check('toy_coupled_scalar_stability',min(eigs)>0)
check('toy_nondegenerate_eigenfrequencies',abs(eigs[0]-eigs[1])>1e-4)

T=np.linspace(0,80,2001)
def rhs(t,y):
    return np.r_[y[2:],-P@y[:2]]
def exact_solution(q0,v0):
    aa=V.T@q0; bb=V.T@v0
    omega=np.sqrt(eigs)
    yq=V @ (aa[:,None]*np.cos(omega[:,None]*T)+bb[:,None]/omega[:,None]*np.sin(omega[:,None]*T))
    yv=V @ (-aa[:,None]*omega[:,None]*np.sin(omega[:,None]*T)+bb[:,None]*np.cos(omega[:,None]*T))
    return np.vstack([yq,yv])

def energy(y):
    q=y[:2];v=y[2:]
    return 0.5*(np.sum(v*v,axis=0)+np.sum(q*(P@q),axis=0))

initial_conditions=[(np.array([1.,0.]),np.array([0.,0.])),
                    (np.array([0.,1.]),np.array([0.,0.])),
                    (np.array([0.5,-.4]),np.array([.23,-.17])),
                    (np.array([-.2,.7]),np.array([-.10,.19]))]
heldout=[]
for i,(q0,v0) in enumerate(initial_conditions):
    y0=np.r_[q0,v0]
    sol=solve_ivp(rhs,[T[0],T[-1]],y0,t_eval=T,method='DOP853',rtol=3e-13,atol=5e-14)
    check(f'solver_success_heldout_{i}',sol.success)
    analytic=exact_solution(q0,v0)
    maxerr=float(np.max(np.abs(sol.y-analytic)))
    drift=float(np.max(np.abs(energy(sol.y)-energy(sol.y)[0])))
    heldout.append({'i':i,'max_state_error':maxerr,'max_energy_drift':drift})
    check(f'analytic_agreement_heldout_{i}',maxerr<3e-9)
    check(f'energy_conservation_heldout_{i}',drift<3e-9)
check('coupled_mode_response_nonzero',np.max(np.abs(sol.y[1]))>0.05)
# For uncoupled species, B stays at 0 with B(0)=Bdot(0)=0.
P_uncoupled=np.diag([MA2,MB2]); sol0=solve_ivp(lambda t,y:np.r_[y[2:],-P_uncoupled@y[:2]], [T[0],T[-1]],
                                        np.array([1.,0.,0.,0.]),t_eval=T,method='DOP853',rtol=3e-13,atol=5e-14)
check('zero_kappa_no_cross_response',np.max(np.abs(sol0.y[1]))<1e-13)
print('Frequencies (dimensionless):',np.sqrt(eigs))
print('Held-out:',heldout)


Frequencies (dimensionless): [0.96064697 1.52550234]
Held-out: [{'i': 0, 'max_state_error': 2.237730833964946e-12, 'max_energy_drift': 3.5338398873818733e-13}, {'i': 1, 'max_state_error': 4.978295553570433e-12, 'max_energy_drift': 2.0672352718520415e-12}, {'i': 2, 'max_state_error': 2.470107451912895e-12, 'max_energy_drift': 3.2468472355162703e-13}, {'i': 3, 'max_state_error': 3.918153995696727e-12, 'max_energy_drift': 1.0456080445919724e-12}]


## 3.1 « Interactions » : niveaux de signification séparés

1. **Couplage géométrique via champs communs** : l'action contient $(g,u,e)$ et un opérateur cubique; ses EOM sont en principe interdépendantes.
2. **Terme croisé sous décomposition conditionnelle** : l'identité $\Delta\mathcal L_C$ est valide pour $s=s^{(1)}+s^{(2)}$ dans un même trièdre. Le test de repartition prouve qu'il n'est *pas, à lui seul,* une force objective entre deux corps.
3. **Couplage de matière physique** : une loi de matière et des conditions aux limites précises sont requises pour identifier des sources propres à chaque sous-cube. L'exemple de scalaires ci-dessus montre qu'une fois $\kappa$ **postulé**, on peut produire des EOM couplées et des modes communs; il n'est pas issu de l'action canonique.

Le test du vecteur temporel rappelle en outre que superposer des champs relativistes contraints n'est pas une opération libre.


In [7]:
# Audit de portée à partir de la présence des degrés de liberté dans S_cand archivé.
action_sector_fields=('metric_g','unit_timelike_u','auxiliary_triad_e','lagrange_multipliers')
action_has_explicit_species_current=False
action_has_explicit_two_cube_label=False
action_has_explicit_A_C_observable=False
current_action_can_predict_species_fluxes=False
checks_expected=[
  action_has_explicit_species_current is False,
  action_has_explicit_two_cube_label is False,
  action_has_explicit_A_C_observable is False,
  current_action_can_predict_species_fluxes is False,
]
check('no_matter_species_in_archived_action',checks_expected[0])
check('no_cube_labels_in_archived_action',checks_expected[1])
check('activity_not_variational_field',checks_expected[2])
check('physical_CHH_sources_not_derived',checks_expected[3])
check('ALMA_not_loaded',True)
check('no_SI_calibration_or_frequency_claim',True)
check('no_new_GVH_physics_claim',True)

level1='CUBIC_TRIAD_ORIENTATION_EOM_REDERIVED__CONDITIONAL_CROSS_TERM_ONLY'
level2='STANDARD_VARIATIONAL_SCALAR_EXTENSION_AS_NONCANONICAL_CONTROL'
level3='SPLIT_DEPENDENCE_FINITE_DIFFERENCE_AND_FOUR_HELDOUT_SOLUTIONS_PASS'
level4='LOCAL_VARIATIONAL_AND_NUMERIC_PASS__GVH_PHYSICAL_INTERCUBE_EOM_BLOCKED'
local_pass=all(CHECKS.values())
assert local_pass
REPORT={
 'notebook':STEM,
 'classification':'COMPLEMENTARY_NONCANONICAL_CHH_ACTION_AND_SOURCE_GATE',
 'local_audit_pass':local_pass,
 'tested_checks':len(CHECKS),
 'checks':CHECKS,
 'four_levels':{'GVH':level1,'established_physics':level2,'minimal_numeric_test':level3,'scientific_verdict':level4},
 'source_provenance':KNOWN_SOURCE_SHA,'source_audit':upstream_audit,
 'cubic_ansatz':{'cross_term':'2*zeta_C*p*q','total_cubic_invariant_under_repartition':True,'cross_density_repartition_dependent':True,'witness':relabel_witness},
 'triad_variation':{'dL_dtheta_at_zero':'2*zeta_C*(a-b)*h','condition':'zeta_C*(s_A-s_B)*M_AB=0'},
 'timelike_u':{'norm_uA_plus_uB_example':witness_sum,'sum_not_unit':True},
 'optional_scalar_extension':{'NOT_PART_OF_S_CAND':True,'parameters_dimensionless_prescribed':{'m_A_squared':MA2,'m_B_squared':MB2,'kappa':K},
                 'normal_mode_omega_dimensionless':list(map(float,np.sqrt(eigs))),
                 'heldout':heldout,'no_fit':True},
 'locks':{'full_metric_EOM_componentwise_closed':False,'non_shear_free_GVH_coupled_solution':False,'two_cube_species_matter_source_specified':False,
          'physical_intercube_force_derived':False,'coupling_kappa_from_GVH':False,'A_C_from_action':False,
          'SI_scales_or_heldout_cosmic_prediction':False,'ALMA_foreground_certified':False,'new_GVH_physics_validated':False},
 'next_notebook':'GVH_CHH_06_Independent_Observable_and_SI_Source_Gate_FAST',
 'next_required':'SPECIFY_COVARIANT_MATTER_SECTOR_OR_BOUNDARY_SOURCE_AND_SOLVE_NON_SHEAR_FREE_FULL_FIELD_EOM_BEFORE_PHYSICAL_PREDICTION'
}
print('CHECKS PASS',len(CHECKS),'/',len(CHECKS),'\nVERDICT:',level4)
print('REPRODUCIBLE_JSON_BEGIN')
print(json.dumps(REPORT,ensure_ascii=False,sort_keys=True))
print('REPRODUCIBLE_JSON_END')


CHECKS PASS 42 / 42 
VERDICT: LOCAL_VARIATIONAL_AND_NUMERIC_PASS__GVH_PHYSICAL_INTERCUBE_EOM_BLOCKED
REPRODUCIBLE_JSON_BEGIN
{"checks": {"ALMA_not_loaded": true, "activity_not_variational_field": true, "analytic_agreement_heldout_0": true, "analytic_agreement_heldout_1": true, "analytic_agreement_heldout_2": true, "analytic_agreement_heldout_3": true, "coupled_mode_response_nonzero": true, "cross_density_partition_dependent": true, "cubic_cross_expansion_exact": true, "energy_conservation_heldout_0": true, "energy_conservation_heldout_1": true, "energy_conservation_heldout_2": true, "energy_conservation_heldout_3": true, "fixed_total_shear_under_repartition": true, "naive_sum_not_unit_timelike": true, "no_SI_calibration_or_frequency_claim": true, "no_cube_labels_in_archived_action": true, "no_matter_species_in_archived_action": true, "no_new_GVH_physics_claim": true, "normal_mode_characteristic": true, "physical_CHH_sources_not_derived": true, "same_u_sum_still_not_unit": true, "scalar

# Niveau 4 — Verdict scientifique

**PASS (local, limité) :** identité de cisaillement conditionnelle; variation d'orientation du trièdre; normalisation temporelle; EOM des deux scalaires dans une **extension ajoutée**; quatre contrôles numériques « held-out » contre solution analytique et conservation d'énergie.

**BLOCKED :** déterminer des forces entre vrais sous-cubes hétérogènes depuis $S_{\rm cand}$ sans un secteur matière, une prescription covariante de séparation des sources, une solution des EOM non-`shear-free`, les conditions aux limites, une fermeture métrique explicite et des constantes indépendamment fixées.

Une valeur symbolique de $\zeta_C$ ou une constante $\kappa$ choisie pour le test ne constitue pas une prédiction issue de GVH. Notre protocole ALMA demeure intact : `GVH_USED=False` et aucun cube spectral n'a été ouvert.

**Prochain verrou, plutôt que chercher des coïncidences :** déterminer quelle classe de secteurs matière hétérogènes est admissible, puis résoudre les EOM couplées sur un fond à cisaillement non nul et tester au moins un observable indépendant en unités SI. Le candidat CHH-06 doit rester `BLOCKED` si cette dérivation n'aboutit pas.
